In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.2 Bytes and byte-level BPE

Byte-level BPE starts from the 256 raw bytes (so *any* text round-trips, no
OOV) and merges frequent adjacent pairs into new tokens. Fewer tokens, same
text.

In [ ]:
from pocketlm import BPETokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
bpe = BPETokenizer().train(corpus[:20000], 400)
text = "to be or not to be, that is the question"
ids = bpe.encode(text)
print("raw bytes:", len(text.encode("utf-8")), " bpe tokens:", len(ids))
print("round trip ok:", bpe.decode(ids) == text)

Byte-level means even an emoji round-trips, it is just several byte tokens.
Merges never lengthen the sequence, so BPE is a strict win on length.

In [ ]:
assert bpe.decode(ids) == text
assert len(ids) <= len(text.encode("utf-8"))
assert bpe.decode(bpe.encode("\U0001f600")) == "\U0001f600"